# Gold reference data
Lookup tables not present in the PSX files: sector names (PSX Data Portal sector list) and SBP policy rate decisions.

In [0]:
%run ../00_utils

In [0]:
sectors = [
    ("0801", "Automobile Assembler", "Equity"), ("0802", "Automobile Parts & Accessories", "Equity"),
    ("0803", "Cable & Electrical Goods", "Equity"), ("0804", "Cement", "Equity"),
    ("0805", "Chemical", "Equity"), ("0806", "Close-End Mutual Fund", "Closed-End Fund"),
    ("0807", "Commercial Banks", "Equity"), ("0808", "Engineering", "Equity"),
    ("0809", "Fertilizer", "Equity"), ("0810", "Food & Personal Care Products", "Equity"),
    ("0811", "Glass & Ceramics", "Equity"), ("0812", "Insurance", "Equity"),
    ("0813", "Inv. Banks / Inv. Cos. / Securities Cos.", "Equity"), ("0814", "Jute", "Equity"),
    ("0815", "Leasing Companies", "Equity"), ("0816", "Leather & Tanneries", "Equity"),
    ("0818", "Miscellaneous", "Equity"), ("0819", "Modarabas", "Equity"),
    ("0820", "Oil & Gas Exploration Companies", "Equity"), ("0821", "Oil & Gas Marketing Companies", "Equity"),
    ("0822", "Paper, Board & Packaging", "Equity"), ("0823", "Pharmaceuticals", "Equity"),
    ("0824", "Power Generation & Distribution", "Equity"), ("0825", "Refinery", "Equity"),
    ("0826", "Sugar & Allied Industries", "Equity"), ("0827", "Synthetic & Rayon", "Equity"),
    ("0828", "Technology & Communication", "Equity"), ("0829", "Textile Composite", "Equity"),
    ("0830", "Textile Spinning", "Equity"), ("0831", "Textile Weaving", "Equity"),
    ("0832", "Tobacco", "Equity"), ("0833", "Transport", "Equity"),
    ("0834", "Vanaspati & Allied Industries", "Equity"), ("0835", "Woollen", "Equity"),
    ("0836", "Real Estate Investment Trust", "REIT"), ("0837", "Exchange Traded Funds", "ETF"),
    ("0838", "Property", "Equity"), ("0839", "Apparel", "Equity"),
]

## So we are creating a reference table here to store two kind of info here i.e. Sector Names (for BI) and SBP policy rates
ref_sector = (spark.createDataFrame(sectors, "sector_code STRING, sector_name STRING, security_type STRING")
                   .withColumn("load_timestamp", F.current_timestamp()))
ref_sector.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.ref_sector")
print(spark.table("workspace.gold.ref_sector").count(), "sectors")

In [0]:
from pyspark.sql.window import Window

# (effective date, policy rate %)
# 2023+ : SBP "Structure of Interest Rates" | 2015-2022 dates: SBP OVR-Repo-History.pdf (rates to verify)
policy = [
    ("2014-11-17", 9.50),
    ("2015-01-26", 8.50), ("2015-03-24", 8.00), ("2015-05-25", 6.50), ("2015-09-14", 6.00),
    ("2016-05-23", 5.75),
    ("2018-01-29", 6.00), ("2018-05-28", 6.50), ("2018-07-16", 7.50), ("2018-10-01", 8.50), ("2018-12-03", 10.00),
    ("2019-02-01", 10.25), ("2019-04-01", 10.75), ("2019-05-21", 12.25), ("2019-07-17", 13.25),
    ("2020-03-18", 12.50), ("2020-03-25", 11.00), ("2020-04-17", 9.00), ("2020-05-18", 8.00), ("2020-06-26", 7.00),
    ("2021-09-21", 7.25), ("2021-11-22", 8.75), ("2021-12-15", 9.75),
    ("2022-04-08", 12.25), ("2022-05-24", 13.75), ("2022-07-13", 15.00), ("2022-11-28", 16.00),
    ("2023-01-24", 17.00), ("2023-03-03", 20.00), ("2023-04-05", 21.00), ("2023-06-27", 22.00),
    ("2024-06-11", 20.50), ("2024-07-30", 19.50), ("2024-09-13", 17.50), ("2024-11-05", 15.00), ("2024-12-17", 13.00),
    ("2025-01-28", 12.00), ("2025-05-06", 11.00), ("2025-12-16", 10.50),
    ("2026-04-28", 11.50),
]

w = Window.orderBy("effective_date")
ref_policy = (spark.createDataFrame(policy, "effective_date STRING, policy_rate_pct DOUBLE")
    .withColumn("effective_date", F.to_date("effective_date"))
    .withColumn("previous_rate_pct", F.lag("policy_rate_pct").over(w))
    .withColumn("rate_change_bps",
                F.round((F.col("policy_rate_pct") - F.col("previous_rate_pct")) * 100).cast("int"))
    .withColumn("direction", F.when(F.col("rate_change_bps") > 0, "Hike")
                              .when(F.col("rate_change_bps") < 0, "Cut")
                              .otherwise("Start"))
    .withColumn("load_timestamp", F.current_timestamp()))

ref_policy.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.ref_policy_rate")
print(spark.table("workspace.gold.ref_policy_rate").count(), "policy decisions")

In [0]:
display(spark.table("workspace.gold.ref_sector"))
display(spark.table("workspace.gold.ref_policy_rate").orderBy("effective_date"))